![ChatWithAgent](ChatWithAgent.png)

#### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential

load_dotenv()
project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model = os.getenv("MODEL_DEPLOYMENT_NAME")

print("Project Endpoint: ", project_endpoint)
print("Model: ", model)

#### Creating the Foundry Client and Agent

In [ ]:
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential
from azure.ai.projects.aio import AIProjectClient

async def create_agent():
    credential = AzureCliCredential()

    # creating the Foundry Project Client
    project_client = AIProjectClient(
        endpoint=project_endpoint,
        credential=credential
    )

    # creating a conversation using the OpenAI Client
    openai_client = project_client.get_openai_client()
    conversation = await openai_client.conversations.create()
    conversation_id = conversation.id
    print("Conversation ID: ", conversation_id)

    # creating the Azure AI Client to interact with the Agent in Foundry
    agent_client = AzureAIClient(
        project_client = project_client,
        conversation_id = conversation_id,
        model_deployment_name=model
    )

    # creating an agent in Foundry
    agent = agent_client.create_agent(
        name="BatmanAgent",
        instructions="You are Batman, the dark knight of Gotham City."
    )
    return agent, credential, agent_client

agent, credential, agent_client = await create_agent()

#### Chatting with the Agent - Non Streaming

In [ ]:

async def non_streaming_example():
    query = "Who is the Joker?"
    result = await agent.run(query)
    print("Agent Response:", result)

await non_streaming_example()